# Question 2: Regional Classification 

Can you predict a country's region/development tier using only its death ratio (Global North vs. South, life-expectancy tier, GNI quartile).

In [1]:
import pandas as pd

from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans

from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

from sklearn.metrics import confusion_matrix

In [ ]:
# merged_df is the fully cleaned working dataframe. It is made in the first question file.
merged_df = pd.read_csv("merged-df.csv")

In [3]:
merged_df.head()

,Unnamed: 0,Country/Territory,Code,Year,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Total Chronic Deaths,GNI_per_capita,HDI,LifeExpectancy
0,0,Afghanistan,AFG,1990,1116,371,44899,11580,2108,3709,5945,2673,34,72435,2684.550019,0.273,45.9672
1,1,Afghanistan,AFG,1991,1136,374,45492,11796,2120,3724,6050,2728,41,73461,2276.289409,0.279,46.6631
2,2,Afghanistan,AFG,1992,1162,378,46557,12218,2153,3776,6223,2830,48,75345,2059.868084,0.287,47.5955
3,3,Afghanistan,AFG,1993,1187,384,47951,12634,2195,3862,6445,2943,56,77657,1525.533426,0.297,51.4664
4,4,Afghanistan,AFG,1994,1211,391,49308,12914,2231,3932,6664,3027,63,79741,1087.961890,0.292,51.4945


First, we perform some initial data preparation to further clean the data from `merged-df.csv`.

In [ ]:
merged_df.drop("Unnamed: 0", axis = 1, inplace = True)

In [5]:
len(merged_df)

5520

We drop any entries containing an N/A value, since these could interfere with our classification and clustering models. Each entry represents an independent "snapshot" rather than part of a time series, so removing entries only reduces the amount of available training data; it does not bias the model toward any particular prediction.

In [6]:
merged_df = merged_df.drop(merged_df[merged_df.isna().any(axis=1)].index)

In [7]:
len(merged_df)

4905

We calculate the death ratio for use in predicting a country's "category."

From this point on, "category" refers to whether a country is classified as "Global North" or "Global South." As defined in our written report, a "Global North" country has a Human Development Index (HDI) of 0.8 or greater, while a "Global South" country has an HDI below 0.8.

In [8]:
chronic_causes = ["Alzheimer's Disease and Other Dementias",
                  "Parkinson's Disease",'Cardiovascular Diseases',
                  'Neoplasms','Diabetes Mellitus','Chronic Kidney Disease',
                  'Chronic Respiratory Diseases','Cirrhosis and Other Chronic Liver Diseases', 'HIV/AIDS']

death_ratio = merged_df[chronic_causes].divide(merged_df['Total Chronic Deaths'], axis = 0)

In [9]:
death_ratio

,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS
0,0.015407,0.005122,0.619852,0.159867,0.029102,0.051205,0.082074,0.036902,0.000469
1,0.015464,0.005091,0.619267,0.160575,0.028859,0.050694,0.082357,0.037135,0.000558
2,0.015422,0.005017,0.617918,0.162161,0.028575,0.050116,0.082593,0.037561,0.000637
3,0.015285,0.004945,0.617472,0.162690,0.028265,0.049732,0.082993,0.037897,0.000721
4,0.015187,0.004903,0.618352,0.161949,0.027978,0.049310,0.083571,0.037960,0.000790
...,...,...,...,...,...,...,...,...,...
5515,0.011099,0.003165,0.245083,0.164297,0.046753,0.031031,0.040496,0.028793,0.429282
5516,0.011500,0.003283,0.253936,0.171894,0.048862,0.032385,0.041800,0.029416,0.406924
5517,0.011994,0.003425,0.263948,0.180358,0.050879,0.033725,0.043277,0.030822,0.381571
5518,0.012594,0.003596,0.276590,0.190698,0.053560,0.035485,0.045132,0.032158,0.350189


We convert HDI values into a "Global North" or "Global South" label, which we use to verify our models' performance.

In [10]:
merged_df["Category"] = merged_df.apply(lambda entry: "Global North" if entry["HDI"] >= 0.8 else "Global South", axis = 1)

In [11]:
merged_df.head()

,Country/Territory,Code,Year,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Total Chronic Deaths,GNI_per_capita,HDI,LifeExpectancy,Category
0,Afghanistan,AFG,1990,1116,371,44899,11580,2108,3709,5945,2673,34,72435,2684.550019,0.273,45.9672,Global South
1,Afghanistan,AFG,1991,1136,374,45492,11796,2120,3724,6050,2728,41,73461,2276.289409,0.279,46.6631,Global South
2,Afghanistan,AFG,1992,1162,378,46557,12218,2153,3776,6223,2830,48,75345,2059.868084,0.287,47.5955,Global South
3,Afghanistan,AFG,1993,1187,384,47951,12634,2195,3862,6445,2943,56,77657,1525.533426,0.297,51.4664,Global South
4,Afghanistan,AFG,1994,1211,391,49308,12914,2231,3932,6664,3027,63,79741,1087.961890,0.292,51.4945,Global South


In [12]:
# List number of country-year entries placed in each category
len(merged_df[merged_df["Category"] == "Global North"]), len(merged_df[merged_df["Category"] == "Global South"])

(1193, 3712)

A grid search would be too cumbersome for this dataset, so we instead use a simple train-test split. For now, we use a 10-Nearest Neighbors model, since our goal is to test whether the death ratio is a practical predictor of a country's category rather than to identify the best possible predictor.

# K-Nearest Neighbors Analysis (K = 10, 80-20 Train-Test Split)


In [13]:
X_training, X_test, y_training, y_test = train_test_split(
    death_ratio,
    merged_df["Category"],
    train_size = 0.8,
    random_state = 50
)

In [14]:
knn_classifier = KNeighborsClassifier(n_neighbors = 10, metric = "euclidean")
knn_classifier.fit(X_training, y_training)

KNeighborsClassifier(metric='euclidean', n_neighbors=10)

In [15]:
y_test_predicted = knn_classifier.predict(X_test)

In [16]:
# Verifying the predictor worked as expected.
y_test_predicted[:10]

array(['Global South', 'Global South', 'Global South', 'Global North',
       'Global South', 'Global South', 'Global South', 'Global South',
       'Global South', 'Global South'], dtype=object)

In [17]:
accuracy_knn = accuracy_score(y_test, y_test_predicted)
precision_south = precision_score(y_test == "Global South", y_test_predicted == "Global South")
precision_north = precision_score(y_test == "Global North", y_test_predicted == "Global North")
recall_south = recall_score(y_test == "Global South", y_test_predicted == "Global South")
recall_north = recall_score(y_test == "Global North", y_test_predicted == "Global North")
f1_south = recall_score(y_test == "Global South", y_test_predicted == "Global South")
f1_north = recall_score(y_test == "Global North", y_test_predicted == "Global North")

knn_confusion_matrix = pd.DataFrame(
    confusion_matrix(y_test, y_test_predicted),
    index = merged_df["Category"].unique(),
    columns = merged_df["Category"].unique()
)

# Print performance of our model
print("Performance of the 10-NN model with 80-20 train-test split")
print("Accuracy:", accuracy_knn)
print("Precision (Global South):", precision_south)
print("Precision (Global North):", precision_north)
print("Recall (Global South):", recall_south)
print("Recall (Global North):", recall_north)
print("F1 Score (Global South):", f1_south)
print("F1 Score (Global North):", f1_north)

Performance of the 10-NN model with 80-20 train-test split
Accuracy: 0.9612640163098879
Precision (Global South): 0.9785522788203753
Precision (Global North): 0.9063829787234042
Recall (Global South): 0.9707446808510638
Recall (Global North): 0.9301310043668122
F1 Score (Global South): 0.9707446808510638
F1 Score (Global North): 0.9301310043668122


In [18]:
# Note that the COLUMNS are predicted values, and the rows are ACTUAL values
knn_confusion_matrix

,Global South,Global North
Global South,213,16
Global North,22,730


Our classifier achieves a high success rate, with an accuracy of around 96%. However, the precision, recall, and F1 scores for the "Global North" class are noticeably lower. This is likely because "Global North" countries make up a larger share of the test set than of the full dataset, which is itself dominated by "Global South" countries. Since the train-test split already performs well, we do not believe cross-validation is necessary here.

# KMeans Clustering (Average Death Ratio by Category)

In [19]:
X_train = death_ratio
y_train = merged_df["Category"]

model = KMeans(n_clusters = 2, init = "random", random_state = 75)

In [20]:
model.fit(X_train)

KMeans(init='random', n_clusters=2, random_state=75)

In [21]:
centroids = model.cluster_centers_
clusters = model.labels_

In [22]:
centroids, clusters

(array([[0.0329517 , 0.00813181, 0.48622694, 0.2403999 , 0.05406688,
         0.04155918, 0.07006321, 0.04330329, 0.0232971 ],
        [0.01106553, 0.0029402 , 0.24552812, 0.12060094, 0.04233323,
         0.02806856, 0.04765843, 0.0444875 , 0.45731749]]),
 array([0, 0, 0, ..., 1, 1, 1], dtype=int32))

Our KMeans clustering shows that "Global South" countries have a higher death ratio for chronic diseases such as cardiovascular disease and HIV/AIDS, while "Global North" countries have a lower death ratio for the same diseases.

In [23]:
death_ratio.join(merged_df["Category"])[death_ratio.join(merged_df["Category"])["Category"] == "Global North"]

,Alzheimer's Disease and Other Dementias,Parkinson's Disease,Cardiovascular Diseases,Neoplasms,Diabetes Mellitus,Chronic Kidney Disease,Chronic Respiratory Diseases,Cirrhosis and Other Chronic Liver Diseases,HIV/AIDS,Category
57,0.043220,0.012034,0.632169,0.231309,0.008501,0.016079,0.039891,0.016694,0.000102,Global North
58,0.044040,0.012111,0.632019,0.230758,0.008508,0.016064,0.039886,0.016515,0.000100,Global North
59,0.044889,0.012140,0.631682,0.230321,0.008567,0.016105,0.039896,0.016301,0.000098,Global North
100,0.044444,0.012698,0.317460,0.469841,0.019048,0.022222,0.076190,0.028571,0.009524,Global North
101,0.046012,0.012270,0.315951,0.469325,0.018405,0.024540,0.076687,0.027607,0.009202,Global North
...,...,...,...,...,...,...,...,...,...,...
5335,0.054693,0.013056,0.376892,0.378146,0.033365,0.036776,0.082098,0.017800,0.007175,Global North
5336,0.055497,0.013134,0.375268,0.377528,0.033633,0.037531,0.082544,0.017616,0.007249,Global North
5337,0.055696,0.013098,0.376124,0.377814,0.033610,0.037144,0.081739,0.017708,0.007068,Global North
5338,0.056027,0.013122,0.375832,0.378380,0.033586,0.037161,0.081321,0.017687,0.006884,Global North
